# Tree-CoT-RT for ASQP — Kaggle

Notebook runs the Tree-CoT-RT pipeline: type-specific multi-path Tree-CoT generation (S1-a), tree-supervised LoRA initialization (S1-b), rule-reward GRPO and offline chain selection (S2), reasoning-aware LoRA fine-tuning (S3), then self-consistent test inference and exact quadruple F1 (S4).

**Before running:** attach Kaggle Datasets containing this project (`pyproject.toml`, `src/tree_cot_rt/`) and `train.jsonl`, `dev.jsonl`, `test.jsonl`; enable a GPU and Internet. Defaults run all supplied records. Generation is local (Qwen2.5-7B-Instruct) instead of the external ChatGPT/Gemini API used by the paper, so it requires no API key but may take many hours. Save the notebook output files from `/kaggle/working/tree_cot_rt_outputs/`.

In [ ]:
# Kaggle Input paths; None means auto-detect beneath /kaggle/input.
PROJECT_DIR = None
DATA_DIR = None
OUTPUT_DIR = "/kaggle/working/tree_cot_rt_outputs"

# Paper defaults. Lower these and set MAX_* for a smoke test only.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
FINAL_MODEL_NAME = "Qwen/Qwen2.5-7B"
CHAINS_PER_TYPE = 10
GRPO_STEPS = 10
SELECTED_CHAINS_PER_TYPE = 2
FINAL_STEPS = 10
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 2
LEARNING_RATE = 5e-5
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 2048
INFERENCE_SAMPLES = 5
MAX_TRAIN_RECORDS = None
MAX_TEST_RECORDS = None
SEED = 444

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

input_root = Path("/kaggle/input")
if not input_root.is_dir():
    raise FileNotFoundError("/kaggle/input was not found; run this notebook on Kaggle.")

def find_project(root):
    matches = [p for folder in root.iterdir() if folder.is_dir() for p in folder.glob("pyproject.toml") if (p.parent / "src" / "tree_cot_rt").is_dir()]
    if PROJECT_DIR:
        candidate = Path(PROJECT_DIR)
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "tree_cot_rt").is_dir():
            return candidate
        raise FileNotFoundError(f"PROJECT_DIR is not a Tree-CoT-RT project: {candidate}")
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one project under /kaggle/input, found {matches}. Set PROJECT_DIR.")
    return matches[0].parent

def find_data(root):
    if DATA_DIR:
        candidate = Path(DATA_DIR)
        if all((candidate / f"{split}.jsonl").is_file() for split in ("train", "dev", "test")):
            return candidate
        raise FileNotFoundError(f"DATA_DIR must contain train.jsonl, dev.jsonl, and test.jsonl: {candidate}")
    matches = [p for folder in root.iterdir() if folder.is_dir() for p in folder.glob("train.jsonl") if all((p.parent / f"{s}.jsonl").is_file() for s in ("dev", "test"))]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one train/dev/test dataset under /kaggle/input, found {matches}. Set DATA_DIR.")
    return matches[0].parent

project_dir = find_project(input_root)
data_dir = find_data(input_root)
sys.path.insert(0, str(project_dir / "src"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Project:", project_dir)
print("Data:", data_dir)
print("Output:", OUTPUT_DIR)
print("Installing project dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "accelerate>=1.0.0", "peft>=0.13.0", "transformers>=4.46.0"])
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before running the full training pipeline.")

In [ ]:
from tree_cot_rt.data import read_jsonl
from tree_cot_rt.training import train

train_records = read_jsonl(data_dir / "train.jsonl")
dev_records = read_jsonl(data_dir / "dev.jsonl")
test_records = read_jsonl(data_dir / "test.jsonl")
if MAX_TRAIN_RECORDS is not None:
    train_records = train_records[:MAX_TRAIN_RECORDS]
if MAX_TEST_RECORDS is not None:
    test_records = test_records[:MAX_TEST_RECORDS]
print(f"Records: train={len(train_records)}, dev={len(dev_records)}, test={len(test_records)}")

training_metrics = train(
    train_records,
    dev_records,
    OUTPUT_DIR,
    model_name=MODEL_NAME,
    final_model_name=FINAL_MODEL_NAME,
    chains_per_type=CHAINS_PER_TYPE,
    grpo_steps=GRPO_STEPS,
    selected_chains_per_type=SELECTED_CHAINS_PER_TYPE,
    final_steps=FINAL_STEPS,
    batch_size=BATCH_SIZE,
    gradient_accumulation=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    max_length=MAX_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
)
training_metrics

In [ ]:
from tree_cot_rt.cli import exact_f1
from tree_cot_rt.data import write_jsonl
from tree_cot_rt.training import predict

predictions = predict(
    test_records,
    Path(OUTPUT_DIR) / "final",
    base_model=FINAL_MODEL_NAME,
    samples=INFERENCE_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
)
prediction_path = Path(OUTPUT_DIR) / "test_predictions.jsonl"
write_jsonl(prediction_path, predictions)
gold = {row.instance_id: [quad.as_dict() for quad in row.quads] for row in test_records}
predicted = {row["instance_id"]: row["quadruples"] for row in predictions}
evaluation = exact_f1(gold, predicted)
evaluation_path = Path(OUTPUT_DIR) / "evaluation_metrics.json"
evaluation_path.write_text(__import__("json").dumps(evaluation, indent=2) + "\n", encoding="utf-8")
print("Exact quadruple micro-F1:", evaluation)
print("Predictions:", prediction_path)
print("Training log:", Path(OUTPUT_DIR) / "training_metrics.json")